## 「192-1: 県を4色に塗り分けよう」 1問目

In [8]:
%matplotlib inline
from pulp import PULP_CBC_CMD, lpSum, lpDot, value
from ortoolpy import model_min, addbinvars
from japanmap import adjacent, pref_map

In [9]:
colors = ["red", "blue", "green", "yellow"]

In [10]:
m = model_min()

In [11]:
vlst = addbinvars(47, 4)

In [13]:
for i in range(1, 48):
    m += lpSum(vlst[i -1]) == 1
    for j in adjacent(i):
        for c in range(4):
            m += vlst[i -1][c] + vlst[j -1][c] <= 1

In [14]:
m.solve(PULP_CBC_CMD(msg=False))

1

In [15]:
cols = [colors[int(value(lpDot(range(4), v)))] for v in vlst]

In [16]:
pref_map(range(1, 48), cols=cols, width=3, rough=True)

## 「192-1: 県を4色に塗り分けよう」 2問目

In [17]:
%matplotlib inline

In [18]:
import pandas as pd
from pulp import PULP_CBC_CMD, lpDot, lpSum, value
from ortoolpy import model_min, addbinvars, addvals
from japanmap import adjacent, pref_map

colors = ["red", "blue", "green", "yellow"]


In [20]:
df = pd.DataFrame(
    [(p, c) for p in range(1, 48) for c in range(4)], columns=['県', '色'])

addbinvars(df)  # 変数の列Varの追加

df[:2]


,県,色,Var
0,1,0,v000377
1,1,1,v000378


In [21]:
m = model_min()

for i, dfi in df.groupby('県'):
    m += lpSum(dfi.Var) == 1  # 色の割当て
    for j in adjacent(i):
        dfj = df[df.県 == j]
        for c in range(4):
            # 隣接していたら、同色は1つまで
            m += dfi.Var.iloc[c] + dfj.Var.iloc[c] <= 1

m.solve(PULP_CBC_CMD(msg=False))

addvals(df)  # 結果の列Valの追加

df[df.Val > 0][:2]

,県,色,Var,Val
3,1,3,v000380,1.0
5,2,1,v000382,1.0


In [22]:
cols = [colors[c] for c in df[df.Val > 0].色]

pref_map(range(1, 48), cols=cols, width=3, rough=True)